# 20 · Adapter dùng chung ViVQA + ViTextVQA — SH (W11)

**Vấn đề:** Hiện có hai adapter chuyên biệt: C1/C3 cho câu hỏi thường (ViVQA) và B3/B3-full cho câu hỏi đọc chữ (ViTextVQA). Demo không biết trước câu hỏi thuộc loại nào, nên với hai adapter thì phải có bộ chọn adapter. Câu hỏi đặt ra: **một adapter train trộn hai dataset có kém adapter chuyên biệt cùng điều kiện không?** (đã hứa trên slide Review 2)

**Thiết kế (luật ghi trong `docs/W11/README.md` TRƯỚC khi chạy, phương án S2):**
- Dữ liệu: ViVQA `train.json` (10.799 câu, **cùng file C1**) + ViTextVQA tập con 8k (8.000 câu, **cùng file B3**) = **18.799 câu**, xáo lẫn.
- **Mọi câu** dùng prompt `ocr` (VietOCR · v1 · 600 · 0). Ảnh không có chữ thì prompt trùng hệt zero-shot.
- Siêu tham số **y hệt C1 và B3** (r 16 · α 32 · q/k/v/o, lr 2e-4, 1 epoch, batch hiệu dụng 16, seed 42, bf16, 4-bit, L4).

| So sánh (ghép cặp) | Mốc | Biên không kém δ = 1,5 điểm EM |
|---|---|---|
| ViVQA test (3.001 câu) | **C1**: EM 73,78 | cận dưới CI95 ΔEM > −1,5 → **KHÔNG KÉM** |
| ViTextVQA test2000 | **B3**: EM 49,25 | cận trên CI95 < −1,5 → **KÉM** · còn lại → **CHƯA KẾT LUẬN** |

Cả hai **KHÔNG KÉM** → **ADAPTER DÙNG CHUNG ĐẠT** (prototype dùng 1 adapter). Ngược lại → **KHÔNG ĐẠT** (prototype dùng 2 adapter chuyên biệt).

| Phần | Việc | Thời gian (L4) |
|---|---|---|
| Train | ≈ 1.175 step × ≈ 18,5 s | ≈ 6 giờ |
| Eval | ViVQA test 3.001 câu + ViTextVQA test2000 | ≈ 1,2 giờ |
| Kết luận | ghép cặp + bootstrap | ≈ 1–2 phút (CPU) |

**Cần trước:** notebook 19 (T4) đã OCR xong ảnh ViVQA và có biên bản `experiments/W11_ocr_cache_vivqa/` trên Drive.

**Chia phiên:** train lưu checkpoint mỗi 100 step (≈ 30 phút); eval ghi mỗi lô 500 câu. Nếu bị ngắt: mở lại notebook, **Run all**. Phần đã xong được bỏ qua, phần dở chạy tiếp.

## 1. Thiết lập môi trường

Giống notebook 18 (cùng pin `transformers==5.17.0`, cùng GPU L4 với C1 và B3). Dừng hẳn nếu GPU không phải L4, vì đổi GPU là đổi điều kiện so sánh.

*Kỳ vọng: commit mới nhất (có notebook 20), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/shared_adapter.py'), 'Code cũ -> git pull chưa lấy được commit W11 bước 4'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với C1/B3). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn ≈ 7 giờ GPU

- **(a) Dữ liệu:** `vivqa/train.json` trùng sha256 với file C1 đã train; tập con 8k trùng sha256 với file B3 đã train.
- **(b) Cấu hình OCR:** đọc từ `ocr_selection.json` (W08), **không gõ tay**.
- **(c) OCR cache ViVQA:** trùng sha256 biên bản W11 (notebook 19). **OCR cache ViTextVQA:** trùng biên bản W10. Hai cache phải phủ đủ ảnh train + test cần dùng. Chưa có biên bản W11 thì dừng và chạy notebook 19 trước.
- **(d) Run mốc trên Drive:** C1, B3 (train_log + predictions), C3, B3-full (predictions, chỉ để mô tả).
- **(e) Selftest 5 script:** `train_qlora` (phải có `mix (W11) OK`), `lora_targets`, `run_baseline`, `shared_adapter`, `full_eval_table`.

*Kỳ vọng: 2 dòng `… sha256 = … ✅`, `paddledet_vietocr | v1 | 600 | 0.0`, 2 dòng `OCR cache … = biên bản ✅`, 4 dòng phủ ảnh dạng `x/x`, 6 dòng run mốc `✅`, `✅ selftest OK (5 script)`. Lần đầu giải nén ảnh ViTextVQA mất khoảng 3–5 phút.*

In [ ]:
import json, hashlib, subprocess
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
VV, VT = 'vivqa', 'vitextvqa_official'
VV_TRAIN_REL = 'data/vivqa/train.json'
VT_TRAIN_REL = 'data/vitextvqa_official/train_subset8000_byimage_seed42.json'
VV_TEST = f'{DRIVE_ROOT}/data/vivqa/test.json'
VT_TEST2000 = f'{DRIVE_ROOT}/data/vitextvqa_official/test_subset2000_seed42.json'
SAVE_STEPS = 100        # checkpoint ≈ 30 phút/lần (không ảnh hưởng chữ ký run)
CHUNK = 500

SH_TRAIN = 'W11_SH_qlora_ocr_vietocr_v1_r16_qwen25vl_shared_vivqa_sub8k_ep1_bf16_L4_seed42'
SH_EV_V = 'W11_SH_eval_qlora_ocr_vietocr_v1_r16_shared_ep1_vivqa_test_bf16_L4_seed42'
SH_EV_T = 'W11_SH_eval_qlora_ocr_vietocr_v1_r16_shared_ep1_vitextvqa_test2000_bf16_L4_seed42'
OUT = f'{EXP_ROOT}/W11_shared_adapter_L4_seed42'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (không chạy eval trên adapter hỏng)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

from src.eval.shared_adapter import REF
meta_of = lambda run: json.load(open(f'{EXP_ROOT}/{run}/train_log.json', encoding='utf-8'))['meta']

# (a) cùng file train với C1 (ViVQA) và B3 (ViTextVQA)
c1_meta, b3_meta = meta_of(REF['C1_train']), meta_of(REF['B3_train'])
assert sha256_file(f'{DRIVE_ROOT}/{VV_TRAIN_REL}') == c1_meta['data_file_sha256'], 'vivqa/train.json KHÁC file C1 -> dừng, báo Claude'
assert sha256_file(f'{DRIVE_ROOT}/{VT_TRAIN_REL}') == b3_meta['data_file_sha256'], 'tập con 8k KHÁC file B3 -> dừng, báo Claude'
print(f"ViVQA train sha256 = C1 ✅ ({c1_meta['num_samples']} câu) | ViTextVQA 8k sha256 = B3 ✅ ({b3_meta['num_samples']} câu)")

# (b) cấu hình OCR đóng băng (W08)
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

# (c) OCR cache: ViVQA = biên bản W11 (notebook 19), ViTextVQA = biên bản W10; phủ đủ ảnh
MAN_VV = f'{EXP_ROOT}/W11_ocr_cache_vivqa/ocr_cache_manifest.json'
MAN_VT = f'{EXP_ROOT}/W10_ocr_cache_fulltrain/ocr_cache_manifest.json'
if not os.path.exists(MAN_VV):
    raise SystemExit('⏸ Chưa có biên bản OCR ViVQA -> chạy notebook 19 (T4) trước, rồi Run all lại notebook này.')
C_VV, C_VT = default_cache_path(DRIVE_ROOT, VV, BACKEND), default_cache_path(DRIVE_ROOT, VT, BACKEND)
assert sha256_file(C_VV) == json.load(open(MAN_VV, encoding='utf-8'))['sha256'], 'OCR cache ViVQA KHÁC biên bản W11 -> dừng, báo Claude'
assert sha256_file(C_VT) == json.load(open(MAN_VT, encoding='utf-8'))['sha256_after_w10'], 'OCR cache ViTextVQA KHÁC biên bản W10 -> dừng, báo Claude'
print('OCR cache ViVQA = biên bản W11 ✅ | OCR cache ViTextVQA = biên bản W10 ✅')
IMG_VV, IMG_VT = prepare_images(VV, DRIVE_ROOT), prepare_images(VT, DRIVE_ROOT)
have_vv, have_vt = set(load_ocr_cache(C_VV)['images']), set(load_ocr_cache(C_VT)['images'])
for lab, f, img, have in (('ViVQA train', f'{DRIVE_ROOT}/{VV_TRAIN_REL}', IMG_VV, have_vv), ('ViVQA test', VV_TEST, IMG_VV, have_vv),
                          ('ViTextVQA 8k', f'{DRIVE_ROOT}/{VT_TRAIN_REL}', IMG_VT, have_vt), ('ViTextVQA test2000', VT_TEST2000, IMG_VT, have_vt)):
    need = set(images_of([f], img))
    print(f'{lab:18s} {len(need & have)}/{len(need)} ảnh có OCR')
    assert need <= have, f'{lab}: OCR cache thiếu ảnh -> chạy lại notebook 19 / báo Claude'

# (d) run mốc trên Drive
for k in ('C1_train', 'B3_train'):
    assert os.path.exists(f'{EXP_ROOT}/{REF[k]}/train_log.json'), f'Thiếu {REF[k]}/train_log.json trên Drive'
    print(f'{k:9s} train_log ✅')
for k, n in (('C1', 3001), ('B3', 2000), ('C3', 3001), ('B3full', 10028)):
    m = len(json.load(open(f'{EXP_ROOT}/{REF[k]}/predictions.json', encoding='utf-8')))
    assert m == n, (k, m)
    print(f'{k:9s} {m} câu ✅')

# (e) selftest
checks = {'src.adaptation.train_qlora': 'mix (W11) OK', 'src.adaptation.lora_targets': 'lora audit OK',
          'src.eval.run_baseline': 'SELFTEST OK', 'src.eval.shared_adapter': 'shared adapter rules OK',
          'src.eval.full_eval_table': 'full_eval_table selftest OK'}
for mod, must in checks.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (5 script)')

## 3. Train SH (≈ 6 giờ)

Dùng **`--mix`** thay cho `--dataset`. Mỗi dataset tự dùng OCR cache của nó. Nếu thiết lập OCR của hai cache lệch nhau, code dừng ngay.

Sau khoảng 4 phút nạp model, log phải có:
- `vivqa: 10799 mẫu (loại 0)` và `vitextvqa_official: 8000 mẫu (loại 0)`;
- `TỔNG 18799 mẫu train, đã xáo lẫn`;
- `LoRA audit: 144 lớp {'llm': 144}`;
- `lịch: 1175 step tổng`.

Sau khi train, `check_sh_train` kiểm **toàn bộ điều kiện hợp lệ** của luật W11: đúng file C1 + B3, siêu tham số = C1 = B3, LoRA chỉ ở LLM, 7.372.800 tham số, và **phần OCR ViTextVQA trùng dấu vân tay B3**. Sai điều nào thì dừng.

**Giữ máy không ngủ và tab mở.** Bị ngắt thì **Run all** lại, train tự chạy tiếp từ checkpoint gần nhất.

*Kỳ vọng: `== XONG: 1175 step …`, `✔ SH train_log hợp lệ …`. **Điểm dừng 1** (an toàn để tắt máy nếu cần).*

In [ ]:
from src.eval.shared_adapter import check_sh_train
if os.path.exists(f'{EXP_ROOT}/{SH_TRAIN}/train_log.json'):
    print('⏭ SH train: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.adaptation.train_qlora --run_name {SH_TRAIN} --epochs 1 --save_steps {SAVE_STEPS} '
            f'--mix {VV}={VV_TRAIN_REL} --mix {VT}={VT_TRAIN_REL} {OCR_ARGS}')

meta = meta_of(SH_TRAIN)
errs = check_sh_train(meta, c1_meta, b3_meta)
assert not errs, 'RUN SH KHÔNG HỢP LỆ -> dừng, gửi Claude:\n' + '\n'.join(errs)
AD_SH = f'{ADP_ROOT}/{SH_TRAIN}/epoch_1'
assert os.path.isdir(AD_SH), f'Thiếu adapter {AD_SH}'
c = meta['cost']
parts_txt = ', '.join('%s %d' % (p['dataset'], p['num_samples']) for p in meta['data_parts'])
print(f"✔ SH train_log hợp lệ: {meta['num_samples']} câu ({parts_txt}) | "
      f"{meta['training']['total_steps']} step | {c['train_runtime_sec'] / 3600:.2f} h (phiên cuối) | {c['sec_per_step']} s/step | "
      f"VRAM {c['peak_vram_gb']} GB | loss {c['final_loss']} | chạy tiếp từ {meta['resumed_from']}")
print('   % mẫu có dòng OCR:', {d: s['pct_samples_with_text'] for d, s in meta['ocr']['stats_by_dataset'].items()})

## 4. Eval SH trên ViVQA test (≈ 35 phút)

Adapter `epoch_1`, prompt chế độ `ocr` với OCR cache ViVQA. Đây đúng là luật mà demo sẽ dùng. Mỗi lô 500 câu được ghi lên Drive ngay khi xong.

*Kỳ vọng: dòng `OCR: …/data/vivqa/ocr/paddledet_vietocr.json`, 7 lô, `== KẾT QUẢ: EM=…`. **Điểm dừng 2.***

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{SH_EV_V}/metrics.json'):
    print('⏭ SH eval ViVQA: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {VV} --data_file {VV_TEST} --run_name {SH_EV_V} '
            f'--chunk {CHUNK} --adapter {AD_SH} {OCR_ARGS}')
m = json.load(open(f'{EXP_ROOT}/{SH_EV_V}/metrics.json', encoding='utf-8'))
print(f"✔ SH ViVQA test: EM={m['metrics']['exact_match']:.4f} ANLS={m['metrics']['anls']:.4f}  (C1: EM 0.7378 ANLS 0.7566)")

## 5. Eval SH trên ViTextVQA test2000 (≈ 40 phút)

Cùng file test2000, cùng OCR cache với lần chấm B3. Bước kết luận sẽ kiểm dấu vân tay OCR trùng B3.

*Kỳ vọng: dòng `OCR: …`, 4 lô, `== KẾT QUẢ: EM=…`. **Điểm dừng 3.***

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{SH_EV_T}/metrics.json'):
    print('⏭ SH eval test2000: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {VT} --data_file {VT_TEST2000} --run_name {SH_EV_T} '
            f'--chunk {CHUNK} --adapter {AD_SH} {OCR_ARGS}')
m = json.load(open(f'{EXP_ROOT}/{SH_EV_T}/metrics.json', encoding='utf-8'))
print(f"✔ SH ViTextVQA test2000: EM={m['metrics']['exact_match']:.4f} ANLS={m['metrics']['anls']:.4f}  (B3: EM 0.4925 ANLS 0.7038)")

## 6. Kết luận theo luật ghi trước

`src/eval/shared_adapter.py` làm ba việc:
1. **Kiểm hợp lệ** train_log và 2 run eval: cùng file test, cùng OCR với B3, adapter đúng run SH. Sai điều nào thì in ❌ và **không** kết luận.
2. **So sánh chính** C1 → SH (ViVQA) và B3 → SH (test2000): ΔEM + CI95, McNemar p, ΔANLS. Từ đó ra kết luận theo biên δ = 1,5 điểm.
3. **Chỉ báo cáo:**
   - tách ViVQA thành nhóm *có / không có dòng OCR*;
   - tỉ lệ "chép chữ OCR" của C1 và SH;
   - so với C3 / B3-full (khác điều kiện, chỉ mô tả);
   - chi phí.

*Kỳ vọng: `✔ kiểm hợp lệ …`, 2 dòng so sánh chính, dòng `➜ KẾT LUẬN THEO LUẬT …`, 2 dòng tách nhóm ViVQA, 2 dòng chép OCR, 2 dòng mô tả, dòng chi phí, file `shared_adapter_summary.json`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
run_cmd(f'python -m src.eval.shared_adapter --exp_root {EXP_ROOT} --sh_train {SH_TRAIN} --sh_vivqa {SH_EV_V} '
        f'--sh_vitext {SH_EV_T} --vivqa_test {VV_TEST} --vivqa_ocr_cache {C_VV} --out {OUT}/shared_adapter_summary.json')

## 7. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy L4 …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy L4 để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 8. ⏸ DỪNG

Gửi Claude output **mục 6** (và dòng `✔ SH train_log hợp lệ …` ở mục 3 nếu còn trong tab). Nếu lỡ đóng tab mất output thì không sao: mọi kết quả nằm trong `experiments/W11_SH*` và `W11_shared_adapter_L4_seed42` trên Drive, Cursor chép và kiểm lại được.